
# 📘 Day 4 学习笔记：自回归生成与 KV Cache 深度解密

> 学习主线：**Prefill / Decode → 自回归逐 token 生成 → KV Cache 为什么存在 → KV Cache 显存公式 → 为什么长上下文会 OOM → Streaming / SSE 工程映射**

---

## 🎯 今日学习大纲

### 1. 你今天要解决的核心问题
- 什么叫 **自回归生成（autoregressive generation）**？
- 为什么大模型不是一次性生成整句，而是 **一个 token 一个 token 地往外吐**？
- **Prefill** 和 **Decode** 到底有什么区别？
- 为什么推理时必须有 **KV Cache**？
- 为什么 **缓存 K/V，不缓存 Q**？
- 为什么上下文一拉长，显存会线性爆炸？
- 为什么很多服务框架（Transformers / vLLM / TGI）都要围绕 cache 做优化？

### 2. 今日完成目标
学完这份 Notebook，你应该能做到：

1. 画出 **Prefill / Decode** 的矩阵与数据流。  
2. 精确说出 **KV Cache 的显存公式**。  
3. 能解释 **为什么上下文从 8K 拉到 32K 会直接 OOM**。  
4. 能写一个 **KV Cache 显存计算器**。  
5. 能回答面试官连续追问：
   - 为什么只缓存 K/V？
   - 没有 KV Cache 会怎样？
   - Prefill 和 Decode 的瓶颈分别是什么？
   - GQA 为什么对 KV Cache 有帮助？
6. 能把今天的底层知识映射到：
   - Streaming
   - Callback
   - SSE / WebSocket
   - token-by-token 输出链路

### 3. 今日掌握技能
- **理论技能**：理解自回归生成与 KV Cache
- **公式技能**：会推 KV Cache 显存公式
- **代码技能**：写 KV Cache calculator + toy incremental decode demo
- **工程技能**：理解 Streaming 输出链路为何天然按 token 驱动

---

## 🧭 今日建议学习节奏
- 第 1 轮：先通读 Markdown 解释，建立全局认知
- 第 2 轮：逐个运行代码单元，看图、看表、改参数
- 第 3 轮：自己手改 `num_layers / seq_len / batch_size / num_kv_heads`
- 第 4 轮：晚上把内容总结成一页面试笔记


In [ ]:

# Day 4 所需基础库
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams['figure.figsize'] = (8, 4.5)
plt.rcParams['axes.unicode_minus'] = False

print('环境就绪 ✅')



---

# 1️⃣ 先建立直觉：什么是自回归生成？

## 自回归（Autoregressive）是什么意思？

大模型做文本生成时，不是“一次把整句话全算出来”，而是：

```text
已知前文 → 预测下一个 token
把新 token 拼回前文 → 再预测下一个 token
再拼回去 → 再预测下一个 token
```

也就是：

```text
x_1, x_2, ..., x_T  →  预测 x_{T+1}
x_1, x_2, ..., x_T, x_{T+1}  →  预测 x_{T+2}
...
```

这就是 **自回归生成**。

---

## 为什么必须逐 token 生成？

因为当前 token 的概率分布依赖于前面的上下文：

\[
P(x_1, x_2, ..., x_n) = \prod_{t=1}^{n} P(x_t \mid x_{<t})
\]

这条公式的含义是：

- 整个序列的联合概率，被拆成很多个“条件概率”的连乘；
- 第 \(t\) 个 token 的预测，依赖于前面所有 token；
- 所以生成阶段天然是 **一步一步** 的。

---

## 因此，大模型推理被拆成两个阶段

### 阶段 A：Prefill
用户把完整 prompt 一次性喂进去，例如：

```text
“请你解释什么是 KV Cache，并举例说明。”
```

模型会一次性处理整段 prompt，计算每一层所有 token 的 Q / K / V。

### 阶段 B：Decode
接下来模型开始生成回复：

```text
“KV” → “Cache” → “是” → ...
```

此时每一步只新增 **1 个 token**，但它仍要和之前所有历史 token 建立注意力关系。

这也是为什么 Decode 阶段天然很慢：

> **每次只多一个 token，但这个 token 要看所有历史。**



---

# 2️⃣ Prefill / Decode 的矩阵维度流动

设：

- \(B\) = batch size
- \(T\) = 当前上下文长度
- \(d_{model}\) = hidden size
- \(H_q\) = query heads 数
- \(H_{kv}\) = key/value heads 数
- \(d_h\) = 单个 head 的维度

并满足：

\[
d_{model} = H_q \cdot d_h
\]

---

## 2.1 Prefill 阶段

输入：

\[
X \in \mathbb{R}^{B \times T \times d_{model}}
\]

经过线性层后：

\[
Q \in \mathbb{R}^{B \times H_q \times T \times d_h}
\]

\[
K \in \mathbb{R}^{B \times H_{kv} \times T \times d_h}
\]

\[
V \in \mathbb{R}^{B \times H_{kv} \times T \times d_h}
\]

在 Prefill 中，整段 prompt 一次性算完，所以吞吐高，GPU 利用率一般更高。

---

## 2.2 Decode 阶段

假设此时已经有历史长度 \(T\)，当前只新进来 1 个 token。

那么新 token 的：

\[
Q_{new} \in \mathbb{R}^{B \times H_q \times 1 \times d_h}
\]

\[
K_{new} \in \mathbb{R}^{B \times H_{kv} \times 1 \times d_h}
\]

\[
V_{new} \in \mathbb{R}^{B \times H_{kv} \times 1 \times d_h}
\]

然后需要和历史缓存拼接：

\[
K_{cache} \in \mathbb{R}^{B \times H_{kv} \times T \times d_h}
\]

\[
V_{cache} \in \mathbb{R}^{B \times H_{kv} \times T \times d_h}
\]

拼接后：

\[
K_{all} \in \mathbb{R}^{B \times H_{kv} \times (T+1) \times d_h}
\]

\[
V_{all} \in \mathbb{R}^{B \times H_{kv} \times (T+1) \times d_h}
\]

当前 token 的注意力是：

\[
\text{Attn}(Q_{new}, K_{all}, V_{all}) = \text{softmax}\left(\frac{Q_{new}K_{all}^{\top}}{\sqrt{d_h}}\right)V_{all}
\]

注意这里的注意力矩阵形状接近：

\[
[B, H_q, 1, T+1]
\]

也就是说：

> **当前只有 1 个 query，但它要看全部历史 key/value。**


In [ ]:

# 用表格把 Prefill / Decode 的维度直觉再整理一遍
shape_df = pd.DataFrame([
    ['输入 X', '[B, T, d_model]', '完整 prompt 或当前输入'],
    ['Q (prefill)', '[B, H_q, T, d_h]', '整段 prompt 的 query'],
    ['K (prefill)', '[B, H_kv, T, d_h]', '整段 prompt 的 key'],
    ['V (prefill)', '[B, H_kv, T, d_h]', '整段 prompt 的 value'],
    ['Q_new (decode)', '[B, H_q, 1, d_h]', '当前新增 token 的 query'],
    ['K_cache', '[B, H_kv, T, d_h]', '历史缓存的 key'],
    ['V_cache', '[B, H_kv, T, d_h]', '历史缓存的 value'],
    ['Attn Score', '[B, H_q, 1, T+1]', '当前 token 对全部历史的注意力'],
], columns=['张量', '形状', '含义'])
shape_df



---

# 3️⃣ 为什么 KV Cache 必须存在？

## 如果没有 KV Cache，会发生什么？

假设你已经有一段长度为 \(T\) 的上下文，现在要生成下一个 token。

如果 **没有 cache**，那么每一步 decode 都得把前面整段序列重新过一遍模型：

```text
第 1 步：重算长度 T 的所有 token
第 2 步：重算长度 T+1 的所有 token
第 3 步：重算长度 T+2 的所有 token
...
```

也就是说，历史 token 的 K/V 会被反复重复计算。

这是极大的浪费。

---

## KV Cache 的核心思想

在 prefill 阶段，我们已经算过历史 token 的 K 和 V 了。

那么在 decode 阶段：

- 旧 token 的 K/V **不需要重算**；
- 只需要计算 **新 token 的 Q / K / V**；
- 然后把新的 K/V 追加到 cache；
- 当前 token 的 Q 去查询整个缓存中的 K/V。

所以缓存的是：

```text
每一层的 K
每一层的 V
```

而不是 Q。

---

## 为什么不缓存 Q？

因为：

- 历史 token 的 Q 只在它自己被生成的那个时刻有用；
- 未来 token 并不会再使用旧 token 的 Q；
- 未来 token 只需要拿 **自己的 Q** 去查 **历史的 K/V**。

所以：

> **K/V 是“被别人反复访问的历史索引和内容”，Q 只是“当前这一步的查询请求”。**

这就是“缓存 K/V，不缓存 Q”的本质原因。



---

# 4️⃣ KV Cache 显存公式（今天最关键）

## 单 batch 情况

单 batch 下，KV Cache 的近似显存大小为：

\[
\text{KV Cache Bytes}
= 2 \times L \times T \times H_{kv} \times d_h \times \text{bytes\_per\_element}
\]

其中：

- \(2\)：因为有 K 和 V 两份
- \(L\)：Transformer 层数
- \(T\)：当前上下文长度
- \(H_{kv}\)：KV heads 数量
- \(d_h\)：每个 head 的维度
- `bytes_per_element`：元素字节数（fp16/bf16 通常 2 bytes）

---

## 加上 batch 后

\[
\text{KV Cache Bytes}
= B \times 2 \times L \times T \times H_{kv} \times d_h \times \text{bytes\_per\_element}
\]

所以显存复杂度是：

\[
O(B \times L \times T \times H_{kv} \times d_h)
\]

注意这里特别关键：

> **是 \(H_{kv}\)，不是一定等于 \(H_q\)。**

所以 GQA / MQA 能省显存，就是因为它们把 \(H_{kv}\) 降下来了。

---

## 例子

假设：

- `batch_size = 1`
- `num_layers = 32`
- `seq_len = 8192`
- `num_kv_heads = 8`
- `head_dim = 128`
- `dtype = fp16 = 2 bytes`

则：

\[
\text{KV Cache}
= 1 \times 2 \times 32 \times 8192 \times 8 \times 128 \times 2
\]

\[
\approx 1{,}073{,}741{,}824 \text{ bytes}
\approx 1 \text{ GB}
\]

这还只是：

- batch = 1
- context = 8K
- 只算 cache，不算模型权重、不算激活、不算额外框架开销

所以一旦 context / batch 上去，OOM 很容易发生。


In [ ]:

def kv_cache_bytes(batch_size, num_layers, seq_len, num_kv_heads, head_dim, bytes_per_element=2):
    return batch_size * 2 * num_layers * seq_len * num_kv_heads * head_dim * bytes_per_element


def bytes_to_readable(n_bytes):
    units = ['B', 'KB', 'MB', 'GB', 'TB']
    size = float(n_bytes)
    for unit in units:
        if size < 1024 or unit == units[-1]:
            return f"{size:.2f} {unit}"
        size /= 1024

example = kv_cache_bytes(batch_size=1, num_layers=32, seq_len=8192, num_kv_heads=8, head_dim=128, bytes_per_element=2)
print('示例 KV Cache =', bytes_to_readable(example))


In [ ]:

# 做一个更通用的 KV Cache 计算表
configs = [
    {'name': 'MHA-like', 'B': 1, 'L': 32, 'T': 8192, 'H_kv': 32, 'd_h': 128, 'bytes': 2},
    {'name': 'GQA-like', 'B': 1, 'L': 32, 'T': 8192, 'H_kv': 8,  'd_h': 128, 'bytes': 2},
    {'name': 'MQA-like', 'B': 1, 'L': 32, 'T': 8192, 'H_kv': 1,  'd_h': 128, 'bytes': 2},
]

rows = []
for cfg in configs:
    val = kv_cache_bytes(cfg['B'], cfg['L'], cfg['T'], cfg['H_kv'], cfg['d_h'], cfg['bytes'])
    rows.append({
        '配置': cfg['name'],
        'batch_size': cfg['B'],
        'layers': cfg['L'],
        'seq_len': cfg['T'],
        'kv_heads': cfg['H_kv'],
        'head_dim': cfg['d_h'],
        'KV Cache': bytes_to_readable(val)
    })

pd.DataFrame(rows)


In [ ]:

# 可视化：MHA / GQA / MQA 的 KV Cache 大小对比
labels = ['MHA-like
H_kv=32', 'GQA-like
H_kv=8', 'MQA-like
H_kv=1']
values = [
    kv_cache_bytes(1, 32, 8192, 32, 128, 2) / (1024**3),
    kv_cache_bytes(1, 32, 8192, 8, 128, 2) / (1024**3),
    kv_cache_bytes(1, 32, 8192, 1, 128, 2) / (1024**3),
]

plt.figure(figsize=(8, 4.5))
plt.bar(labels, values)
for i, v in enumerate(values):
    plt.text(i, v + 0.02, f'{v:.2f} GB', ha='center')
plt.ylabel('KV Cache (GB)')
plt.title('同样层数/上下文下，不同 KV Heads 的显存差异')
plt.show()



---

# 5️⃣ 为什么上下文越长越容易 OOM？

从公式里你可以直接看出：

\[
\text{KV Cache Bytes} \propto T
\]

也就是说：

- `8K -> 32K`，KV Cache 变成 4 倍
- `32K -> 128K`，又变成 4 倍

如果 batch 也变大：

\[
\text{KV Cache Bytes} \propto B
\]

于是：

- 上下文翻 4 倍
- batch 再翻 4 倍
- 总 cache 就会翻 **16 倍**

这就是很多服务一上长上下文就崩的直接原因。


In [ ]:

# 画出上下文长度增长时的 KV Cache 曲线
seq_lens = np.array([512, 1024, 2048, 4096, 8192, 16384, 32768, 65536, 131072])
kv_gb = [kv_cache_bytes(1, 32, T, 8, 128, 2) / (1024**3) for T in seq_lens]

plt.figure(figsize=(8, 4.5))
plt.plot(seq_lens, kv_gb, marker='o')
for x, y in zip(seq_lens, kv_gb):
    plt.text(x, y, f'{y:.2f}', fontsize=8)
plt.xscale('log', base=2)
plt.xlabel('Context Length (tokens)')
plt.ylabel('KV Cache (GB)')
plt.title('GQA-like 配置下，KV Cache 随上下文长度线性增长')
plt.grid(alpha=0.3)
plt.show()


In [ ]:

# 再画一个：不同 batch size 的影响
batch_sizes = [1, 2, 4, 8]
seq_lens = np.array([1024, 2048, 4096, 8192, 16384, 32768])

plt.figure(figsize=(8, 4.8))
for B in batch_sizes:
    vals = [kv_cache_bytes(B, 32, T, 8, 128, 2) / (1024**3) for T in seq_lens]
    plt.plot(seq_lens, vals, marker='o', label=f'batch={B}')

plt.xscale('log', base=2)
plt.xlabel('Context Length (tokens)')
plt.ylabel('KV Cache (GB)')
plt.title('KV Cache 同时受 context length 和 batch size 影响')
plt.grid(alpha=0.3)
plt.legend()
plt.show()



---

# 6️⃣ 没有 Cache vs 有 Cache：计算代价差别有多大？

## 6.1 直觉版

假设你已经有长度 \(T_0\) 的 prompt，然后继续生成 \(N\) 个新 token。

### 情况 A：没有 KV Cache
第 1 步要重新算长度 \(T_0\)  
第 2 步要重新算长度 \(T_0+1\)  
第 3 步要重新算长度 \(T_0+2\)  
...

总开销近似像：

\[
\sum_{i=0}^{N-1} (T_0 + i)
\]

甚至如果把全注意力成本也考虑进去，代价会更大。

### 情况 B：有 KV Cache
每一步只新增 1 个 token 的 Q/K/V，并让这个新 Q 去查历史 K/V。

于是你避免了“重算整个历史序列的 K/V”。

---

## 6.2 面试回答要点

如果面试官问：

> **KV Cache 的本质价值是什么？**

你可以回答：

- 不是改变 attention 数学公式；
- 而是避免在自回归 decode 时反复重算历史 token 的 K/V；
- 让每一步只需要计算新 token 的 Q/K/V；
- 因此显著降低推理延迟。


In [ ]:

# 用一个非常粗糙但直观的 operation count 做对比
# 这里只是建立增长直觉，不是严格 FLOPs 公式

def naive_recompute_cost(prompt_len, new_tokens):
    # 每步都重算整段序列，简化为累加长度
    return sum(prompt_len + i for i in range(new_tokens))


def cached_decode_cost(prompt_len, new_tokens):
    # prefill 先一次性处理 prompt，然后每步只处理 1 个新 token
    return prompt_len + new_tokens

prompt_len = 2048
new_tokens_list = np.array([16, 32, 64, 128, 256, 512, 1024])
naive = np.array([naive_recompute_cost(prompt_len, n) for n in new_tokens_list])
cached = np.array([cached_decode_cost(prompt_len, n) for n in new_tokens_list])

plt.figure(figsize=(8, 4.5))
plt.plot(new_tokens_list, naive, marker='o', label='No KV Cache (粗略示意)')
plt.plot(new_tokens_list, cached, marker='o', label='With KV Cache (粗略示意)')
plt.xlabel('Generated Tokens')
plt.ylabel('Relative Compute Cost')
plt.title('没有 Cache vs 有 Cache 的开销直觉对比')
plt.grid(alpha=0.3)
plt.legend()
plt.show()



---

# 7️⃣ Toy Demo：用最小代码验证“增量解码 + KV Cache”

下面我们不用复杂框架，只用 NumPy 写一个 **最小化单头因果自注意力**，来直观看两件事：

1. **全量因果 attention** 的最后一个 token 输出
2. **增量 decode + cache** 的最后一个 token 输出

如果二者一致，就说明：

> 在数学上，incremental decoding + KV Cache 与 full causal attention 是一致的；
> 它只是把“重复计算历史”优化掉了。


In [ ]:

np.random.seed(42)

# 一个很小的 toy sequence
T = 6
d_model = 8

def softmax(x, axis=-1):
    x = x - np.max(x, axis=axis, keepdims=True)
    e = np.exp(x)
    return e / np.sum(e, axis=axis, keepdims=True)

# toy 输入和投影矩阵
X = np.random.randn(T, d_model)
Wq = np.random.randn(d_model, d_model)
Wk = np.random.randn(d_model, d_model)
Wv = np.random.randn(d_model, d_model)

Q = X @ Wq
K = X @ Wk
V = X @ Wv

# 全量因果 attention（只看最后一个 token）
q_last = Q[-1:]                       # [1, d]
K_hist = K[:T]                        # [T, d]
V_hist = V[:T]                        # [T, d]

scores_full = (q_last @ K_hist.T) / math.sqrt(d_model)   # [1, T]
weights_full = softmax(scores_full, axis=-1)             # [1, T]
out_full = weights_full @ V_hist                         # [1, d]

# 增量 decode + cache 思维：假设前 T-1 个 token 的 K/V 已缓存
K_cache = K[:T-1].copy()
V_cache = V[:T-1].copy()

q_new = Q[T-1:T]           # 当前新 token 的 query
k_new = K[T-1:T]           # 当前新 token 的 key
v_new = V[T-1:T]           # 当前新 token 的 value

K_all = np.concatenate([K_cache, k_new], axis=0)
V_all = np.concatenate([V_cache, v_new], axis=0)

scores_cache = (q_new @ K_all.T) / math.sqrt(d_model)
weights_cache = softmax(scores_cache, axis=-1)
out_cache = weights_cache @ V_all

print('full attention 输出是否与 cache decode 一致：', np.allclose(out_full, out_cache, atol=1e-8))
print('
full 输出：
', out_full)
print('
cache 输出：
', out_cache)


In [ ]:

# 可视化最后一个 token 的注意力分布
plt.figure(figsize=(8, 2.8))
plt.bar(np.arange(T), weights_full.flatten())
plt.xlabel('历史 token index')
plt.ylabel('attention weight')
plt.title('最后一个 token 对历史全部 token 的注意力分布')
plt.show()



---

# 8️⃣ Prefill 与 Decode 的性能瓶颈到底不同在哪里？

## Prefill 的特征
- 一次输入很多 token
- 矩阵乘法更大
- GPU 更容易跑满
- 更偏向“吞吐型”问题

## Decode 的特征
- 每次只增加 1 个 token
- 但要反复访问历史 K/V Cache
- 对显存带宽和缓存访问更敏感
- 更偏向“延迟型”问题

---

## 面试官如果追问

### 问：Prefill 和 Decode 谁更吃算力？
回答思路：

- Prefill 一次处理整段 prompt，矩阵乘法规模更大，算力利用更高；
- Decode 每次只有 1 个 query，GPU 更难饱和，但又必须频繁访问整个历史 cache；
- 所以 Decode 往往更受 **memory bandwidth / cache 访问效率 / 调度效率** 影响。

### 问：为什么有时 GPU 利用率不高，但服务还是慢？
回答思路：

- 因为 Decode 阶段常常不是纯算力瓶颈；
- 可能是 KV Cache 访问、调度、batch 组织、网络 streaming 等链路拖慢了整体延迟。


In [ ]:

# 用一个小图帮助你把 Prefill / Decode 的差异视觉化
stages = ['Prefill', 'Decode']
compute_intensity = [9, 4]   # 只是示意，不是真实数值
memory_pressure = [5, 9]     # 只是示意，不是真实数值

x = np.arange(len(stages))
width = 0.35

plt.figure(figsize=(8, 4.2))
plt.bar(x - width/2, compute_intensity, width, label='矩阵乘法/算力压力')
plt.bar(x + width/2, memory_pressure, width, label='缓存/带宽压力')
plt.xticks(x, stages)
plt.ylim(0, 10)
plt.title('Prefill vs Decode：性能瓶颈侧重点不同（示意图）')
plt.legend()
plt.show()



---

# 9️⃣ 工程排障视角：为什么一开长上下文就炸？

## 常见现象

### 现象 1：8K 正常，32K 直接 OOM
可能原因：
- `max_model_len` 拉太高
- batch size 没降
- 模型是 MHA 而不是 GQA
- dtype 仍然是 fp16/bf16 cache

### 现象 2：GPU 利用率不高，但 token/s 很低
可能原因：
- decode 阶段 batch 太小
- cache 访问低效
- streaming / post-processing / tokenizer 在 CPU 侧拖慢
- 请求长度分布极不均匀

### 现象 3：模型越长越慢
可能原因：
- KV Cache 线性变大
- 每个新 token 都要看更长历史
- 带宽和 cache miss 压力加重

---

## 你该怎么排查？

建议按这个顺序：

```text
1. 看模型配置：num_layers / num_attention_heads / num_key_value_heads / head_dim
2. 看服务参数：max_model_len / batch_size / max_num_seqs
3. 算 KV Cache 理论值
4. 看 dtype：fp16 / bf16 / int8 / fp8 cache
5. 看是否存在 GQA / MQA
6. 看 Prefill / Decode latency 分解
7. 看网络 streaming 和 tokenizer 是否拖后腿
```



---

# 🔟 应用工程映射：为什么今天这部分和 Streaming / SSE 强相关？

## 10.1 token-by-token 输出链路

今天学的自回归生成，直接决定了为什么前端看到的是：

```text
“你”
“你好”
“你好，今天”
“你好，今天我们”
...
```

也就是一个 token 一个 token 地流出来。

因为模型底层就是：

```text
decode step 1 -> 产出 token 1
decode step 2 -> 产出 token 2
decode step 3 -> 产出 token 3
```

所以应用层天然可以把每一步的结果通过：

- SSE
- WebSocket
- callback event
- async generator

持续推给前端。

---

## 10.2 典型后端链路

```text
用户请求
   ↓
构造 prompt
   ↓
prefill
   ↓
decode step 1
   ↓  (token event)
SSE / callback 输出
   ↓
decode step 2
   ↓  (token event)
SSE / callback 输出
   ↓
...
   ↓
完成 / stop token
```

所以你今天的知识，会直接映射到这些工程问题：

- 为什么 streaming 是增量的？
- 为什么首 token 延迟（TTFT）很重要？
- 为什么长 prompt 会拖慢首 token？
- 为什么 decode token/s 受 KV Cache 影响很大？

---

## 10.3 如果面试官问：Streaming 的底层基础是什么？

你可以答：

- 底层是 autoregressive decoding；
- 模型逐 token 生成，因此天然可被包装成事件流；
- Prefill 决定首 token 延迟，Decode 决定持续输出速度；
- KV Cache 决定 decode 阶段是否足够高效。


In [ ]:

# 用一个很简化的“事件流”模拟 Streaming 直觉
import time

def fake_streaming_generate(tokens, delay=0.1):
    emitted = ''
    events = []
    for tok in tokens:
        emitted += tok
        events.append({'event': 'token', 'data': tok, 'partial_text': emitted})
        time.sleep(0.01)  # notebook里不要真的太慢
    events.append({'event': 'done', 'data': None, 'partial_text': emitted})
    return events

sample_tokens = ['KV', ' Cache', ' 可以', ' 避免', ' 重算', ' 历史', ' K/V', '。']
events = fake_streaming_generate(sample_tokens)
pd.DataFrame(events)



---

# 1️⃣1️⃣ 今日面试高频问答（建议背到“能自然说出来”）

## Q1：什么是 Prefill？什么是 Decode？
**满分回答思路：**

- Prefill 是把用户的整段 prompt 一次性送入模型，计算所有历史 token 的表示以及每层的 K/V；
- Decode 是在此基础上逐 token 生成，每一步只新增一个 token；
- Prefill 更像吞吐型计算，Decode 更像延迟型计算。

---

## Q2：为什么 KV Cache 只缓存 K/V，不缓存 Q？
**满分回答思路：**

- 历史 token 的 K/V 会被未来每一个新 token 反复访问；
- 历史 token 的 Q 只在它自己生成的那一步有用；
- 所以缓存 K/V 能避免重算历史，而缓存 Q 没有收益。

---

## Q3：KV Cache 的显存复杂度是多少？
**满分回答思路：**

\[
O(B \times L \times T \times H_{kv} \times d_h)
\]

更完整地写成 bytes 公式：

\[
B \times 2 \times L \times T \times H_{kv} \times d_h \times \text{bytes\_per\_element}
\]

---

## Q4：为什么长上下文容易 OOM？
**满分回答思路：**

- 因为 KV Cache 随上下文长度 \(T\) 线性增长；
- 随 batch size 也线性增长；
- 上下文一拉长，显存很快被 cache 占满；
- 尤其是 MHA 比 GQA/MQA 更容易爆。

---

## Q5：Prefill 和 Decode 谁是推理瓶颈？
**满分回答思路：**

- Prefill 处理整段 prompt，更吃大矩阵乘法吞吐；
- Decode 每次只有一个 query，更依赖 cache 访问效率和显存带宽；
- 实际服务中两者都重要，但长期 streaming 体验通常很受 decode 影响。



---

# 1️⃣2️⃣ 今日最小作业

## 你今天至少要完成这 4 件事

### 作业 1：手改 KV Cache 计算器参数
试这些参数并观察变化：
- `seq_len = 4096 / 8192 / 32768 / 131072`
- `batch_size = 1 / 2 / 4 / 8`
- `num_kv_heads = 32 / 8 / 1`
- `dtype = 2 bytes / 1 byte`

### 作业 2：自己写一句话解释
请你不用看笔记，自己口述：

> 为什么 KV Cache 必须存在？

### 作业 3：自己写一句话解释

> 为什么缓存 K/V，而不缓存 Q？

### 作业 4：工程映射
写一个 10 行以内的小结：

> 为什么今天学的自回归生成，会自然映射到 Streaming / SSE？

---

## 建议你今晚整理成一页笔记

```text
今日主题：自回归生成与 KV Cache
核心公式：KV Cache bytes = B × 2 × L × T × H_kv × d_h × bytes
矩阵维度：Q_new / K_cache / V_cache / attention score
显存瓶颈：context length 与 batch size 线性放大 cache
面试问答：为什么只缓存 K/V？为什么长上下文会 OOM？
工程排障：Prefill vs Decode，谁吃吞吐，谁吃带宽
源码映射：generate / cache / streaming callback / SSE response
```



---

# 1️⃣3️⃣ 推荐学习平台 / 视频 / 仓库（只给 3 个高质量资源）

## 资源 1：Andrej Karpathy - Let’s build GPT: from scratch, in code, spelled out
- 类型：YouTube 视频
- 适合用途：建立 Transformer、训练、推理、decode 的完整工程直觉
- 链接：https://www.youtube.com/watch?v=kCc8FmEb1nY

## 资源 2：Andrej Karpathy - Intro to Large Language Models
- 类型：YouTube 视频
- 适合用途：建立大模型整体系统观，理解训练、推理、部署链路
- 链接：https://www.youtube.com/watch?v=zjkBMFhNj_g

## 资源 3：karpathy / nanoGPT
- 类型：GitHub 仓库
- 适合用途：源码级理解 GPT 的最小实现，便于你把今天的 prefill / decode / generate 思想对应到代码
- 链接：https://github.com/karpathy/nanoGPT

---

## 📚 额外阅读（非必看，但强烈推荐）
- Hugging Face 文档：Caching 解释  
  https://huggingface.co/docs/transformers/main/cache_explanation

---

## ✅ 今日一句话总总结

> **自回归生成决定了大模型必须逐 token 解码；KV Cache 的作用是避免重复计算历史 K/V；上下文长度和 batch size 会线性放大 cache 显存；而 Streaming / SSE 本质上就是把 decode 的每一步结果持续输出给前端。**
